## Build a trigram language model

In [1]:
import torch
import matplotlib.pyplot as plt

In [2]:
words = open('names.txt').read().split()

In [3]:
len(words)
# longest word
max(len(w) for w in words)

15

In [4]:
# all characters + '.'
chars = sorted(list(set(''.join(words)))) + ['.']

In [5]:
# string to integer and the other way around
stoi = {c:i for i, c in enumerate(chars)}
itos = {i:c for i, c in enumerate(chars)}

In [6]:
# each two letter combination
combs = []
for i in range(len(chars)):
    for j in range(len(chars)):
        combs.append(chars[i] + chars[j])
# sort them
combs = sorted(combs)
# remove '..' and 'a.'-type combinations
combs = [c for c in combs if c[1] != '.']
# dictionary
stoi2 = {c:i for i, c in enumerate(combs)}
# remove letters followed by '.' 
itos2 = {i:c for i, c in enumerate(combs)}

In [7]:
xs = []
ys = []
for w in words:
    chs = ['.'] + list(w) + ['.']
    for ch1, ch2, ch3 in zip(chs, chs[1:], chs[2:]):
        ix1 = stoi2[ch1+ch2]
        ix2 = stoi[ch3]
        xs.append(ix1)
        ys.append(ix2)
xs = torch.tensor(xs)
ys = torch.tensor(ys)

In [ ]:
# split into 80% train and 10% validation and 10% test

In [8]:
num = xs.nelement()
num

196113

In [9]:
import torch.nn.functional as F

In [16]:
# one hot encoding
xenc = F.one_hot(xs, len(stoi2)).float()

In [17]:
# initialize weights
W = torch.randn(len(stoi2), len(stoi), requires_grad=True).float()

In [19]:
for i in range(200):
    # forward pass 
    logits = xenc @ W
    counts = logits.exp()
    probs = counts / counts.sum(dim=1, keepdim=True)

    # from the probabilities matrix, plug the probability of the correct next string 
    # and take the (mean) negative log likelihood
    loss = -probs[torch.arange(num), ys].log().mean()
    #print every 10th loss
    if i % 10 == 0:
        print(loss.item())

    # backward pass
    W.grad = None
    loss.backward()

    # update weights
    W.data += -50 * W.grad

2.173976182937622
2.1716952323913574
2.1695120334625244
2.1674201488494873
2.165414333343506
2.1634888648986816
2.16163969039917
2.1598615646362305
2.158151149749756
2.156503915786743
2.154916524887085
2.153386116027832
2.151909112930298
2.1504833698272705
2.1491050720214844
2.147772789001465
2.146483898162842
2.145236015319824
2.1440277099609375
2.1428563594818115


In [136]:
ix2 = torch.randint(len(stoi2), (1,)).item()
out = [itos2[ix2][1]]

xenc = F.one_hot(torch.tensor([ix2]), num_classes=len(stoi2)).float()
logits = xenc @ W
p = (logits.exp() / logits.exp().sum(1, keepdims=True))

ix = torch.multinomial(p, num_samples=1, replacement=True).item()
out.append(itos[ix])
out

ix2 = stoi2[itos2[ix2][1] + itos[ix]]
ix2

578

In [160]:
# sample from the neural net
#g = torch.Generator().manual_seed(2147483647)
for i in range(10):
    
    ix2 = 4 #torch.randint(len(stoi2), (1,)).item()
    out = [itos2[ix2][1]]
    while True:
        
        xenc = F.one_hot(torch.tensor([ix2]), num_classes=len(stoi2)).float()
        logits = xenc @ W
        p = (logits.exp() / logits.exp().sum(1, keepdims=True))

        ix = torch.multinomial(p, num_samples=1, replacement=True).item()
        out.append(itos[ix])
        if ix == 26:
            break
        ix2 = stoi2[itos2[ix2][1] + itos[ix]]
       
    print(''.join(out))

el.
evie.
elami.
eshina.
emjna.
edianiestachir.
ellahfnnic.
elo.
emma.
everence.


In [51]:
p.shape

torch.Size([1, 27])